In [2]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import numpy as np
from tqdm.auto import tqdm

In [3]:
#I have 2 partquets math1, math 2 I need to concat them into 1 df
df1 = pd.read_parquet('math_part1.parquet')
df2 = pd.read_parquet('math_part2.parquet')
df_math = pd.concat([df1, df2], ignore_index=True)
print(df_math.shape)
df_math.head()

(12500, 4)


,problem,level,type,solution
0,"Let \[f(x) = \left\{\n\begin{array}{cl} ax+3, ...",Level 5,Algebra,"For the piecewise function to be continuous, t..."
1,A rectangular band formation is a formation wi...,Level 5,Algebra,Let $x$ be the number of band members in each ...
2,What is the degree of the polynomial $(4 +5x^3...,Level 3,Algebra,This polynomial is not written in standard for...
3,Evaluate $\left\lceil3\left(6-\frac12\right)\r...,Level 3,Algebra,"Firstly, $3\left(6-\frac12\right)=18-1-\frac12..."
4,Sam is hired for a 20-day period. On days that...,Level 3,Algebra,Call $x$ the number of days Sam works and $y$ ...


In [4]:
df_aime = pd.read_parquet("aime_clean.parquet")
print(df_aime.shape)
df_aime.head()


(933, 4)


,aime_id,problem,level,solution
0,1983-1,"Let $x$ , $y$ and $z$ all exceed $1$ and let $...",Level 6,60
1,1983-2,"Let $f(x)=|x-p|+|x-15|+|x-p-15|$ , where $0 < ...",Level 6,15
2,1983-3,What is the product of the real roots of the e...,Level 6,20
3,1983-4,A machine-shop cutting tool has the shape of a...,Level 6,26
4,1983-5,Suppose that the sum of the squares of two com...,Level 6,4


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_path = "../../../../scratch/s6019595/models/L1-Qwen3-8B-Max/"
model_LCPO = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map=device
)
tokenizer = AutoTokenizer.from_pretrained(model_path)


`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 2/2 [00:07<00:00,  3.81s/it]
The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


In [6]:
model_LCPO.eval()  

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 4096)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (up_proj): Linear(in_features=4096, out_features=12288, bias=False)
          (down_proj): Linear(in_features=12288, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((4096,), eps=1e-06)
        (post_attention_la

In [14]:
states = []

for index, q in tqdm(df_aime.iterrows(), total=len(df_aime), desc="Processing rows"):
    inputs = tokenizer(q["problem"], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model_LCPO(**inputs, output_hidden_states=True)

    hidden_last = outputs.hidden_states[-1][:, -1, :]
    states.append(hidden_last.detach().to(torch.float16).cpu().numpy())

states = np.stack(states, axis=0)
np.save("hidden_states_aime.npy", states)


Processing rows: 100%|██████████| 933/933 [00:38<00:00, 24.49it/s]
